# grantMinded Demo

In [1]:
# # grantminded_demo_multi.py
# # demo pipeline:
# #   - crawl 3 seed sites (depth up to 3)
# #   - for each page:
# #       1) LR grant/not-grant (bundle)
# #       2) weighted keyword_score + cosine_score to mission text
# #       3) mission filter: keyword_score >= 7 OR (keyword_score > 0 AND cosine >= 0.7)
# #       4) final SVM mission relevance score
# #   - write up to 30 rows per site to a CSV (no corpus files)

# import os
# import re
# import time
# import random
# from urllib.parse import urljoin, urlparse

# import requests
# from bs4 import BeautifulSoup
# import pandas as pd
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.metrics.pairwise import cosine_similarity
# import joblib

# # --------------------------------------------------
# # config / paths
# # --------------------------------------------------
# HEADERS = {
#     "User-Agent": (
#         "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
#         "AppleWebKit/537.36 (KHTML, like Gecko) "
#         "Chrome/114.0.0.0 Safari/537.36"
#     )
# }

# SAVE_DIR = r"C:\Users\miked\Desktop2\IConnectFoundation\grantMinded"
# os.makedirs(SAVE_DIR, exist_ok=True)

# # LR grant/not-grant bundle (with vectorizer + scaler + model)
# GRANT_MODEL_BUNDLE_PATH = os.path.join(SAVE_DIR, "grant_logreg_bundle.pkl")

# # final mission SVM + vectorizer
# SVM_VECTORIZER_PATH = os.path.join(SAVE_DIR, "relevance_vectorizer.joblib")
# SVM_MODEL_PATH      = os.path.join(SAVE_DIR, "relevance_svm_best.joblib")

# # demo csv path
# DEMO_CSV_PATH = os.path.join(SAVE_DIR, "logs", "demo_multi_results.csv")
# os.makedirs(os.path.dirname(DEMO_CSV_PATH), exist_ok=True)

# PER_SITE_TARGET = 30   # <- 30 per site
# MAX_PAGES       = 400  # hard cap across all seeds
# MAX_DEPTH       = 3    # BFS depth

# MAX_SITE_VISITS = 15      # max pages to VISIT per site (pass or fail)
# SVM_THRESHOLD   = 0.0     # require svm_mission_score >= 0


# # --------------------------------------------------
# # seeds: 3 websites to crawl
# # --------------------------------------------------
# SEED_URLS = [
#     "https://www.afar.org/funding-opportunities",
#     "https://www.rwjf.org/en/grants.html",
#     "https://www.gatesfoundation.org/about/how-we-work/grant-opportunities",
# ]

# # --------------------------------------------------
# # mission reference text (for cosine)
# # --------------------------------------------------
# _reference_text = """
# Our mission is to reduce social isolation and improve cognitive and emotional
# health among older adults through technology-assisted conversation, meaningful
# social connection, and community-based interventions, with a focus on aging,
# dementia, Alzheimer's disease, social isolation, telehealth, and community-based
# 501(c)(3) nonprofit work.
# """

# # ============================================================
# # Domain-expert keyword weights (mission relevance)
# # ============================================================
# keyword_weights = {
#     "aging": 3,
#     "dementia": 3,
#     "isolation": 4,
#     "alzheimer's": 3,
#     "telehealth": 2,
#     "501(c)(3)": 2,
#     "grant": 3,
#     "funding": 3,
# }

# # allow-hint keywords for link triage
# allow_hint_keywords = [
#     "grant", "grants", "fund", "funds", "funding",
#     "opportunity", "opportunities", "rfp", "rfa", "rfi",
#     "request-for-proposals", "request for proposals",
#     "apply", "application", "guidelines", "program", "loi", "initiative",
#     "grant-opportunities", "grant-opportunit",
# ]

# deny_list = [
#     "/blog", "/press", "/press-release", "/news",
#     "/stories", "/story",
#     "/team", "/board", "/contact", "/contact-us",
#     "/privacy", "/terms", "/careers", "/jobs", "/event", "/events",
#     "/media", "/newsletter", "/subscribe",
#     "/login", "/sign-in", "/signin", "/sign_in",
#     "/cart", "/donate",
#     "/faq", "frequently-asked-questions", "frequently asked questions",
# ]

# BINARY_EXTS = (
#     ".pdf",".doc",".docx",".xls",".xlsx",".ppt",".pptx",
#     ".zip",".rar",".7z",".png",".jpg",".jpeg",".gif",".svg",
#     ".mp3",".mp4",".mov",".avi",".wav",
# )

# # --------------------------------------------------
# # basic helpers
# # --------------------------------------------------
# def fetch_url(url, timeout=20):
#     try:
#         resp = requests.get(url, headers=HEADERS, timeout=timeout)
#         if resp.status_code == 200 and "text/html" in resp.headers.get("Content-Type", ""):
#             return resp.text
#         else:
#             print(f"  fetch failed ({resp.status_code}) for {url}")
#     except Exception as e:
#         print(f"  error fetching {url}: {e}")
#     return None

# def html_to_text(html):
#     """Strip HTML to clean text and also return soup."""
#     soup = BeautifulSoup(html, "html.parser")
#     for tag in soup(["script", "style", "noscript"]):
#         tag.decompose()
#     txt = soup.get_text(separator=" ")
#     txt = re.sub(r"\s+", " ", txt or "")
#     return txt.strip(), soup

# def same_domain(url, base_url):
#     try:
#         p1 = urlparse(url)
#         p2 = urlparse(base_url)
#         return p1.netloc.lower() == p2.netloc.lower()
#     except Exception:
#         return False

# # -----------------------
# # deny filter
# # -----------------------
# def is_denied_url(u):
#     """
#     Filter by:
#     - binary file extensions
#     - simple path-based deny_list
#     """
#     try:
#         p = urlparse(u)
#     except Exception:
#         return True

#     path = (p.path or "").lower()

#     for ext in BINARY_EXTS:
#         if path.endswith(ext):
#             return True

#     for d in deny_list:
#         d_clean = d.strip().lower()
#         if d_clean and d_clean in path:
#             return True

#     return False

# # -----------------------
# # keyword + cosine
# # -----------------------
# def keyword_score(text, keywords):
#     """
#     Weighted keyword score using your keyword_weights dict.
#     Returns (score, matched_keywords_list).
#     """
#     score = 0
#     matched = []
#     t = (text or "").lower()
#     for word, weight in keywords.items():
#         pat = rf"\b{re.escape(word.lower())}\b"
#         if re.search(pat, t):
#             score += weight
#             matched.append(word)
#     return score, matched

# def _simple_tokenizer(s):
#     s = re.sub(r"[^a-z0-9\s]", " ", (s or "").lower())
#     toks = s.split()
#     return [t for t in toks if len(t) > 2]

# def cosine_score(page_text, ref_text=_reference_text):
#     docs = [ref_text, page_text or ""]
#     vec = TfidfVectorizer(stop_words="english",
#                           tokenizer=_simple_tokenizer,
#                           token_pattern=None)
#     m = vec.fit_transform(docs)
#     sim = cosine_similarity(m[0:1], m[1:2])[0][0]
#     return float(sim)

# # -----------------------
# # link relevance (for crawling)
# # -----------------------
# def looks_relevant_link(href, text_lower):
#     """
#     A link looks promising if allow_hint_keywords appear in href or anchor text.
#     """
#     href_l = (href or "").lower()
#     if any(k in href_l for k in allow_hint_keywords):
#         return True
#     if any(k in (text_lower or "") for k in allow_hint_keywords):
#         return True
#     return False

# def extract_links(current_url, soup, base_url, generic_cap=20):
#     """
#     Prioritize grant-like links, then some generic links.
#     """
#     links_priority, links_generic = [], []
#     if soup is None:
#         return []

#     for a in soup.find_all("a", href=True):
#         href = a.get("href") or ""
#         href_l = href.lower()

#         # skip anchors/mailto/etc
#         if href_l.startswith("#") or href_l.startswith("mailto:") \
#            or href_l.startswith("tel:") or href_l.startswith("javascript:"):
#             continue

#         link = urljoin(current_url, href)
#         if not link.startswith(("http://", "https://")):
#             continue
#         if not same_domain(link, base_url):
#             continue
#         if is_denied_url(link):
#             continue

#         txt = (a.get_text(strip=True) or "").lower()

#         is_pagination = bool(
#             re.search(r"(next|older|previous|more|>>|«|»|‹|›|load more|page\s*\d+)", txt)
#             or re.search(r"(?:[?&](?:page|p|start|offset)=\d+)", link, re.I)
#         )

#         if looks_relevant_link(href, txt) or is_pagination:
#             links_priority.append(link)
#         else:
#             links_generic.append(link)

#     seen, out = set(), []
#     for u in links_priority + links_generic[:generic_cap]:
#         if u not in seen:
#             seen.add(u)
#             out.append(u)
#     return out

# # --------------------------------------------------
# # LR grant/not-grant (from bundle)
# # --------------------------------------------------
# GRANT_VECTORIZER = None
# GRANT_SCALER = None
# GRANT_MODEL = None
# GRANT_THRESHOLD = 0.1  # explicit, per your earlier setup



# try:
#     if os.path.exists(GRANT_MODEL_BUNDLE_PATH):
#         _bundle = joblib.load(GRANT_MODEL_BUNDLE_PATH)
#         GRANT_VECTORIZER = _bundle.get("vectorizer", None)
#         GRANT_SCALER     = _bundle.get("scaler", None)
#         GRANT_MODEL      = _bundle.get("model", None)
#         GRANT_THRESHOLD  = 0.1  # force 0.1 as agreed
#         print(f"Loaded grant LR model bundle from {GRANT_MODEL_BUNDLE_PATH}")
#     else:
#         print(f"WARNING: grant model bundle not found at {GRANT_MODEL_BUNDLE_PATH}")
# except Exception as e:
#     print(f"WARNING: could not load grant model bundle: {e}")

# def is_grant_by_lr(text, threshold=GRANT_THRESHOLD):
#     """
#     Use pretrained LR (TF-IDF + scaler) to decide if text describes a grant.
#     Returns (is_grant_bool, prob).
#     If model missing, falls back to (True, 1.0) so we don't silently drop pages.
#     """
#     if not text or len(text.strip()) < 50:
#         return False, 0.0

#     if GRANT_MODEL is None or GRANT_VECTORIZER is None:
#         # fail-safe: keep the page
#         return True, 1.0

#     X = GRANT_VECTORIZER.transform([text])
#     if GRANT_SCALER is not None:
#         X = GRANT_SCALER.transform(X)
#     prob = float(GRANT_MODEL.predict_proba(X)[0][1])
#     return prob >= threshold, prob

# # --------------------------------------------------
# # SVM relevance model
# # --------------------------------------------------
# def load_svm_model():
#     vec = joblib.load(SVM_VECTORIZER_PATH)
#     clf = joblib.load(SVM_MODEL_PATH)
#     print(f"Loaded relevance_vectorizer + relevance_svm_best from {SAVE_DIR}")
#     return vec, clf

# def compute_svm_score(text, vec, clf):
#     X = vec.transform([text])
#     if hasattr(clf, "decision_function"):
#         score = clf.decision_function(X)[0]
#     elif hasattr(clf, "predict_proba"):
#         score = clf.predict_proba(X)[0, 1]
#     else:
#         score = float(clf.predict(X)[0])
#     return float(score)

# # --------------------------------------------------
# # crawl a single site (BFS up to depth, per-site target, per-site visits)
# # --------------------------------------------------
# def crawl_site(seed_url, svm_vec, svm_clf, rows, visited_global,
#                per_site_target=PER_SITE_TARGET,
#                max_depth=MAX_DEPTH,
#                max_site_visits=MAX_SITE_VISITS):
#     """
#     Crawl one domain starting from seed_url and append rows in-place.
#     Uses BFS with (url, depth).

#     rows: list of row dicts (shared across seeds)
#     visited_global: set of URLs already seen (shared across seeds)

#     Stops after:
#       - collecting per_site_target SVM-passing rows for this site, OR
#       - visiting max_site_visits pages for this site, OR
#       - running out of links in the queue.
#     """
#     queue = [(seed_url, 0)]
#     base_url = seed_url
#     site_rows = 0          # SVM-passing pages for this seed
#     site_visits = 0        # total pages visited (pass or fail)

#     site_host = urlparse(seed_url).netloc.lower()

#     while queue and site_rows < per_site_target and site_visits < max_site_visits:
#         url, depth = queue.pop(0)

#         if url in visited_global:
#             continue
#         visited_global.add(url)

#         if is_denied_url(url):
#             print(f"SKIP denied: {url}")
#             continue

#         if not same_domain(url, base_url):
#             print(f"SKIP other domain: {url}")
#             continue

#         site_visits += 1
#         print(f"\nVISIT (depth={depth}, visits={site_visits}, "
#               f"site_rows={site_rows}, total_rows={len(rows)}): {url}")

#         html = fetch_url(url)
#         time.sleep(random.uniform(0.8, 1.5))

#         if html is None:
#             continue

#         text, soup = html_to_text(html)
#         if not text:
#             continue

#         title = soup.title.string.strip() if soup.title and soup.title.string else ""

#         # 1) LR grant/not-grant
#         is_gr, lr_prob = is_grant_by_lr(text)
#         print(f"  LR prob(grant)={round(lr_prob,3)} -> is_grant={is_gr}")
#         if not is_gr:
#             # still allowed to crawl deeper, we just don't save this page
#             pass_filter = False
#         else:
#             # 2) keyword + cosine mission filter
#             k_score, matched = keyword_score(text, keyword_weights)
#             c_score = cosine_score(text)
#             mission_ok = (k_score >= 7) or (k_score > 0 and c_score >= 0.7)
#             print(
#                 f"  keyword_score={k_score}, matched={matched}, "
#                 f"cosine={round(c_score,3)}, mission_ok={mission_ok}"
#             )

#             if not mission_ok:
#                 pass_filter = False
#             else:
#                 # 3) final SVM mission score
#                 svm_score = compute_svm_score(text, svm_vec, svm_clf)
#                 print(f"  SVM mission score={round(svm_score,3)}")

#                 if svm_score < SVM_THRESHOLD:
#                     print(f"  SVM filter: score {round(svm_score,3)} < {SVM_THRESHOLD} -> DROP")
#                     pass_filter = False
#                 else:
#                     pass_filter = True
#                     snippet = text[:500]
#                     row = {
#                         "site": site_host,
#                         "url": url,
#                         "title": title,
#                         "lr_prob_grant": lr_prob,
#                         "keyword_score": k_score,
#                         "keyword_matched": ";".join(matched),
#                         "cosine_score": c_score,
#                         "svm_mission_score": svm_score,
#                         "snippet": snippet,
#                     }
#                     rows.append(row)
#                     site_rows += 1
#                     print(f"  --> SAVED ROW #{site_rows} for {site_host} "
#                           f"(total_rows={len(rows)})")

#         # expand links if we still have depth and visit budget
#         if depth < max_depth and site_visits < max_site_visits:
#             child_links = extract_links(url, soup, base_url)
#             print(f"  found {len(child_links)} child links")
#             for nxt in child_links:
#                 if nxt not in visited_global and nxt not in [u for (u, _) in queue]:
#                     queue.append((nxt, depth + 1))

#     print(f"Finished site {site_host}: "
#           f"collected {site_rows} rows, visited {site_visits} pages")

# # --------------------------------------------------
# # run demo across all seeds
# # --------------------------------------------------
# def run_demo():
#     print("\nLoading SVM + LR models...")
#     svm_vec, svm_clf = load_svm_model()

#     rows = []
#     visited_global = set()

#     SEEDS = [
#         "https://www.afar.org/funding-opportunities",
#         "https://www.rwjf.org/en/grants.html",
#         "https://www.gatesfoundation.org/about/how-we-work/grant-opportunities"
#     ]

#     for seed in SEEDS:
#         print(f"\n========== CRAWLING SEED: {seed} ==========")
#         crawl_site(
#             seed_url=seed,
#             svm_vec=svm_vec,
#             svm_clf=svm_clf,
#             rows=rows,
#             visited_global=visited_global,
#             per_site_target=PER_SITE_TARGET,   # 30 per site
#             max_depth=MAX_DEPTH,               # BFS depth limit
#             max_site_visits=MAX_SITE_VISITS    # ✅ correct kwarg
#         )

#     if not rows:
#         print("No rows collected; nothing to write.")
#         return

#     df = pd.DataFrame(rows)  # ✅ use pd, not pandas
#     df.to_csv(DEMO_CSV_PATH, index=False, encoding="utf-8")
#     print(f"\nWrote demo CSV with {len(rows)} rows to: {DEMO_CSV_PATH}")


# if __name__ == "__main__":
#     run_demo()


Loaded grant LR model bundle from C:\Users\miked\Desktop2\IConnectFoundation\grantMinded\grant_logreg_bundle.pkl

Loading SVM + LR models...
Loaded relevance_vectorizer + relevance_svm_best from C:\Users\miked\Desktop2\IConnectFoundation\grantMinded

========== CRAWLING SEED: https://www.afar.org/funding-opportunities ==========

VISIT (depth=0, visits=1, site_rows=0, total_rows=0): https://www.afar.org/funding-opportunities
  LR prob(grant)=1.0 -> is_grant=True
  keyword_score=12, matched=['aging', "alzheimer's", 'grant', 'funding'], cosine=0.038, mission_ok=True
  SVM mission score=-0.66
  SVM filter: score -0.66 < 0.0 -> DROP
  found 47 child links

VISIT (depth=1, visits=2, site_rows=0, total_rows=0): https://www.afar.org/grantee-spotlight-interviews-2024
  LR prob(grant)=0.957 -> is_grant=True
  keyword_score=9, matched=['aging', 'grant', 'funding'], cosine=0.027, mission_ok=True
  SVM mission score=-1.213
  SVM filter: score -1.213 < 0.0 -> DROP
  found 65 child links

VISIT (dep

In [3]:
# grantminded_demo_pipeline.py
# demo pipeline (no corpus saving):
#   - crawl static seeds + web search seeds
#   - for each page:
#       1) LR grant/not-grant (bundle)
#       2) weighted keyword_score + cosine_similarity to mission text
#       3) mission filter:
#            keyword_score >= 7
#            OR (keyword_score > 0 AND cosine_similarity >= 0.07)
#       4) final SVM mission relevance score with threshold
#   - do NOT save html/txt files
#   - write up to 300 SVM-passing rows to demo_csv.csv

import os
import re
import time
import random
import hashlib
import datetime as _dt
from urllib.parse import urlparse, urlunparse, urljoin

import requests
from bs4 import BeautifulSoup
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import joblib

# -----------------------
# config / paths
# -----------------------
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/114.0.0.0 Safari/537.36"
    )
}

BASE_DIR   = r"C:\Users\miked\Desktop2\IConnectFoundation\grantMinded"
DEMO_DIR   = os.path.join(BASE_DIR, "demo")
os.makedirs(DEMO_DIR, exist_ok=True)

# output csv (demo)
DEMO_CSV_PATH = os.path.join(DEMO_DIR, "demo_csv.csv")

# LR grant/not-grant bundle (with vectorizer + scaler + model)
GRANT_MODEL_BUNDLE_PATH = os.path.join(BASE_DIR, "grant_logreg_bundle.pkl")

# final mission SVM + vectorizer
SVM_VECTORIZER_PATH = os.path.join(BASE_DIR, "relevance_vectorizer.joblib")
SVM_MODEL_PATH      = os.path.join(BASE_DIR, "relevance_svm_best.joblib")

# caps
MAX_DEMO_ROWS    = 300   # total SVM-passing rows across everything
MAX_DEPTH        = 3     # BFS depth
MAX_SITE_VISITS  = 25    # per site: max pages visited (pass or fail)
PER_SITE_TARGET  = 5     # per site: max SVM-passing rows
GRANT_THRESHOLD  = 0.1   # LR prob threshold (grant vs not)
SVM_THRESHOLD    = 0.0   # SVM threshold (tune if you want stricter filter)

# ============================================================
# Domain-expert keyword weights (mission relevance)
# ============================================================
keyword_weights = {
    "aging": 3,
    "dementia": 3,
    "isolation": 4,
    "alzheimer's": 3,
    "telehealth": 2,
    "501(c)(3)": 2,
    "grant": 3,
    "funding": 3,
}

# ============================================================
# Example web search queries (same as OG corpus)
# ============================================================
search_keywords = [
    "aging nonprofit funding",
    "alzheimer's foundation grant",
    "dementia telehealth grant",
    "community isolation grant 501(c)(3)",
    "social isolation grant",
]

# ============================================================
# Reference text (for cosine similarity baseline)
# ============================================================
_reference_text = (
    "health care programs and partnerships that improve outcomes and quality of life for older adults, "
    "people with disabilities, and caregivers. initiatives to align health and social care, community grants, 501 c 3. "
    "leveraging technology as a tool for connection, promoting meaningful engagement"
)

# ============================================================
# Seeds (static sites)
# ============================================================
static_urls = [
    "https://www.rwjf.org/en/grants/active-funding-opportunities.html",
    "https://www.eda.gov/funding/funding-opportunities",
    "https://www.walmart.org/how-we-give/open-applications",
    "https://www.gatesfoundation.org/about/how-we-work/grant-opportunities",
]

# ============================================================
# deny lists and binary ext
# ============================================================
deny_list = [
    "blog", "press", "press-release", "news",
    "stories", "story",
    "about", "team", "board", "contact", "contact-us",
    "privacy", "terms", "careers", "jobs", "event", "events",
    "media", "newsletter", "subscribe", "search",
    "login", "sign-in", "signin", "sign_in",
    "cart", "donate", "grantee-stories",
    "faq", "frequently asked questions", "frequently-asked-questions",
]

BINARY_EXTS = (
    ".pdf", ".doc", ".docx", ".xls", ".xlsx", ".ppt", ".pptx",
    ".zip", ".rar", ".7z", ".png", ".jpg", ".jpeg", ".gif", ".svg",
    ".mp3", ".mp4", ".mov", ".avi", ".wav",
)

deny_domains = [
    # Paid grant databases / subscription services
    "grantwatch.com",
    "www.grantwatch.com",
    "grantportal.com",
    "www.grantportal.com",
    "thegrantportal.com",
    "www.thegrantportal.com",
    "grantstation.com",
    "www.grantstation.com",
    "instrumentl.com",
    "www.instrumentl.com",
    "grantgopher.com",
    "www.grantgopher.com",
    "grantselect.com",
    "www.grantselect.com",
    "grantforward.com",
    "www.grantforward.com",
    "devex.com",
    "www.devex.com",
    "catholicfundingguide.com",
    "www.catholicfundingguide.com",

    # Other paid or semi-paid grant warehouses / dubious aggregators
    "fundsforngos.org",
    "www.fundsforngos.org",
    "grantadvisor.org",
    "grantdomain.com",
    "www.grantdomain.com",

    # Senior/grants scam-like aggregators
    "grantsforseniors.org",
    "www.grantsforseniors.org",
    "benefits.gov",
    "www.benefits.gov",

    # Sites known to block scraping or require login/payment
    "grantfinder.com",
    "www.grantfinder.com",
    "foundationsearch.com",
    "www.foundationsearch.com",
    "candid.org",
    "www.candid.org",
    "fconlinefoundationcenter.org",
    "www.fconlinefoundationcenter.org",
]

# ============================================================
# allow-hint keywords for link triage
# ============================================================
allow_hint_keywords = [
    "grant", "grants", "fund", "funds", "funding",
    "opportunity", "opportunities", "rfp", "rfa", "rfi",
    "request-for-proposals", "request for proposals",
    "apply", "application", "guidelines", "program", "loi", "initiative",
    "grant-opportunities", "grant-opportunit",
]

# ============================================================
# helpers: ids, time, text clean
# ============================================================
def normalize_url(u):
    p = urlparse(u)
    p = p._replace(fragment="")
    return urlunparse((p.scheme.lower(), p.netloc.lower(), p.path, p.params, p.query, ""))

def doc_id_from_url(u):
    return hashlib.sha1(normalize_url(u).encode("utf-8")).hexdigest()[:16]

def now_iso_utc():
    t = _dt.datetime.now(_dt.timezone.utc).replace(microsecond=0)
    return t.isoformat().replace("+00:00", "Z")

def html_to_clean_text(html):
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    text = soup.get_text(separator=" ", strip=True)
    return re.sub(r"\s+", " ", text).strip(), soup

def content_sha1(text):
    return hashlib.sha1((text or "").encode("utf-8")).hexdigest()

# ============================================================
# keyword + cosine
# ============================================================
def keyword_score(text, keywords):
    score = 0
    matched = []
    t = (text or "").lower()
    for word, weight in keywords.items():
        pat = rf"\b{re.escape(word.lower())}\b"
        if re.search(pat, t):
            score += weight
            matched.append(word)
    return score, matched

def _simple_tokenizer(s):
    s = re.sub(r"[^a-z0-9\s]", " ", (s or "").lower())
    toks = s.split()
    return [t for t in toks if len(t) > 2]

def cosine_score(page_text, ref_text=_reference_text):
    docs = [ref_text, page_text or ""]
    vec = TfidfVectorizer(
        stop_words="english",
        tokenizer=_simple_tokenizer,
        token_pattern=None,
    )
    m = vec.fit_transform(docs)
    sim = cosine_similarity(m[0:1], m[1:2])[0][0]
    return float(sim)

# ============================================================
# LR grant/not-grant loader + predictor
# ============================================================
GRANT_VECTORIZER = None
GRANT_SCALER     = None
GRANT_MODEL      = None

try:
    if os.path.exists(GRANT_MODEL_BUNDLE_PATH):
        _bundle = joblib.load(GRANT_MODEL_BUNDLE_PATH)
        GRANT_VECTORIZER = _bundle.get("vectorizer", None)
        GRANT_SCALER     = _bundle.get("scaler", None)
        GRANT_MODEL      = _bundle.get("model", None)
        print(f"Loaded grant LR model bundle from {GRANT_MODEL_BUNDLE_PATH}")
    else:
        print(f"WARNING: grant model bundle not found at {GRANT_MODEL_BUNDLE_PATH}")
except Exception as e:
    print(f"WARNING: could not load grant model bundle: {e}")

def is_grant_by_lr(text, threshold=GRANT_THRESHOLD):
    """
    Use pretrained logistic regression (TF-IDF + scaler) to decide if text describes a grant.
    Returns (is_grant_bool, prob).
    If model missing, falls back to (True, 1.0) so we don't silently drop pages.
    """
    if not text or len(text.strip()) < 50:
        return False, 0.0

    if GRANT_MODEL is None or GRANT_VECTORIZER is None:
        return True, 1.0

    X = GRANT_VECTORIZER.transform([text])
    if GRANT_SCALER is not None:
        X = GRANT_SCALER.transform(X)
    prob = float(GRANT_MODEL.predict_proba(X)[0][1])
    return prob >= threshold, prob

# ============================================================
# SVM relevance model
# ============================================================
def load_svm_model():
    vec = joblib.load(SVM_VECTORIZER_PATH)
    clf = joblib.load(SVM_MODEL_PATH)
    print(f"Loaded relevance_vectorizer + relevance_svm_best from {BASE_DIR}")
    return vec, clf

def compute_svm_score(text, vec, clf):
    X = vec.transform([text])
    if hasattr(clf, "decision_function"):
        score = clf.decision_function(X)[0]
    elif hasattr(clf, "predict_proba"):
        score = clf.predict_proba(X)[0, 1]
    else:
        score = float(clf.predict(X)[0])
    return float(score)

# ============================================================
# url filters + link extraction
# ============================================================
def _strip_www(host: str) -> str:
    return host[4:] if host.lower().startswith("www.") else host.lower()

def same_domain(u, base):
    try:
        hu = _strip_www(urlparse(u).netloc)
        hb = _strip_www(urlparse(base).netloc)
        return hu == hb or hu.endswith("." + hb) or hb.endswith("." + hu)
    except Exception:
        return False

def is_denied_url(u):
    """
    Unified url filter:
    - blocks binary files (BINARY_EXTS)
    - blocks deny_domains (grantwatch, thegrantportal, etc.)
    - blocks any URL whose PATH contains any phrase from deny_list
    """
    try:
        p = urlparse(u)
    except Exception:
        return True

    host = (p.netloc or "").lower()
    path = (p.path or "").lower()

    for ext in BINARY_EXTS:
        if path.endswith(ext):
            return True

    for dom in deny_domains:
        if host.endswith(dom):
            return True

    for d in deny_list:
        d_clean = d.strip().lower()
        if not d_clean:
            continue
        if d_clean in path:
            return True

    return False

def looks_relevant_link(href, text_lower):
    href_l = (href or "").lower()
    if any(k in href_l for k in allow_hint_keywords):
        return True
    if any(k in (text_lower or "") for k in allow_hint_keywords):
        return True
    return False

def extract_links(current_url, soup, base_url, generic_cap=20):
    links_priority, links_generic = [], []
    if soup is None:
        return []

    for a in soup.find_all("a", href=True):
        href = a.get("href") or ""
        href_l = href.lower()

        if href_l.startswith("#") or href_l.startswith("mailto:") \
           or href_l.startswith("tel:") or href_l.startswith("javascript:"):
            continue

        link = urljoin(current_url, href)
        if not link.startswith(("http://", "https://")):
            continue
        if not same_domain(link, base_url):
            continue
        if is_denied_url(link):
            continue

        txt = (a.get_text(strip=True) or "").lower()

        is_pagination = bool(
            re.search(r"(next|older|previous|more|>>|«|»|‹|›|load more|page\s*\d+)", txt)
            or re.search(r"(?:[?&](?:page|p|start|offset)=\d+)", link, re.I)
        )

        if looks_relevant_link(href, txt) or is_pagination:
            links_priority.append(link)
        else:
            links_generic.append(link)

    seen, out = set(), []
    for u in links_priority + links_generic[:generic_cap]:
        if u not in seen:
            seen.add(u)
            out.append(u)
    return out

# ============================================================
# web search (ddgs / duckduckgo_search / google)
# ============================================================
def _norm(u):
    try:
        p = urlparse(u)
        return f"{p.scheme.lower()}://{p.netloc.lower()}{p.path}".rstrip("/")
    except Exception:
        return u

def _limit_per_domain(urls, max_per_domain=5):
    out, seen = [], {}
    for u in urls:
        host = urlparse(u).netloc.lower()
        seen[host] = seen.get(host, 0) + 1
        if seen[host] <= max_per_domain:
            out.append(u)
    return out

def _seed_looks_promising(url: str, title: str = "", text: str = "") -> bool:
    ul = (url or "").lower()
    tl = (title or text or "").lower()
    return any(tok in ul for tok in allow_hint_keywords) or any(tok in tl for tok in allow_hint_keywords)

def search_web(query, max_results=30):
    urls_raw = []
    q = query.replace("501(c)(3)", '("501(c)(3)" OR 501c3)')

    try:
        from ddgs import DDGS
        with DDGS() as ddgs:
            res = list(ddgs.text(q, max_results=max_results*2))
            urls_raw.extend([(r.get("href"), r.get("title", "")) for r in res if r.get("href")])
        print(f"ddgs returned {len(urls_raw)} raw results for: {query}")
    except Exception as e:
        print(f"ddgs error for '{query}': {e}")
        try:
            from duckduckgo_search import DDGS as OldDDGS
            with OldDDGS() as ddgs:
                res = list(ddgs.text(q, max_results=max_results*2))
                urls_raw.extend([(r.get("href"), r.get("title", "")) for r in res if r.get("href")])
            print(f"duckduckgo_search returned {len(urls_raw)} raw results for: {query}")
        except Exception as e2:
            print(f"duckduckgo_search error for '{query}': {e2}")

    if len(urls_raw) < max_results:
        try:
            from googlesearch import search as gsearch
            g = [(u, "") for u in gsearch(q, num_results=max_results) if u]
            urls_raw.extend(g)
            print(f"googlesearch returned {len(g)} raw results for: {query}")
        except Exception as e:
            print(f"googlesearch blocked/unavailable for '{query}': {e}")

    filtered = []
    for href, title in urls_raw:
        if not href or not href.startswith("http"):
            continue
        if is_denied_url(href):
            continue
        if _seed_looks_promising(href, title):
            filtered.append(_norm(href))

    filtered = list(dict.fromkeys(filtered))
    filtered = _limit_per_domain(filtered, max_per_domain=5)
    random.shuffle(filtered)
    filtered = filtered[:max_results]

    print(f"using {len(filtered)} urls after filters for: {query}")
    return filtered

# ============================================================
# HTTP fetch
# ============================================================
def fetch_url(url, timeout=20):
    if is_denied_url(url):
        print(f"skipping denied url {url}")
        return None
    try:
        resp = requests.get(url, headers=HEADERS, timeout=timeout, allow_redirects=True)
        if resp.status_code == 200 and "text/html" in resp.headers.get("Content-Type", ""):
            return resp.text
        else:
            print(f"  fetch failed ({resp.status_code}) for {url}")
    except Exception as e:
        print(f"  error fetching {url}: {e}")
    return None

# ============================================================
# row builder (no HTML/TEXT files saved)
# ============================================================
def build_demo_row(url, clean_text, k_score, matched_list, c_score, svm_score, scrape_num):
    matched_set = set(matched_list)
    aging       = 1 if "aging" in matched_set else 0
    dementia    = 1 if "dementia" in matched_set else 0
    alzhiemers  = 1 if "alzheimer's" in matched_set else 0
    isolation   = 1 if "isolation" in matched_set else 0
    telehealth  = 1 if "telehealth" in matched_set else 0
    np_501c3    = 1 if "501(c)(3)" in matched_set else 0
    grant_kw    = 1 if "grant" in matched_set else 0
    funding_kw  = 1 if "funding" in matched_set else 0

    did         = doc_id_from_url(url)
    snippet     = clean_text[:2000]
    sha1        = content_sha1(clean_text)
    date_seen   = now_iso_utc()

    row = {
        "url": normalize_url(url),
        "scrape_num": scrape_num,
        "content_snippet": snippet,
        "score": float(k_score + (c_score or 0.0)),
        "matched_keywords": ", ".join(matched_list),
        "num_keywords": len(matched_list),
        "aging": aging,
        "dementia": dementia,
        "alzhiemers": alzhiemers,
        "isolation": isolation,
        "telehealth": telehealth,
        "np_501c3": np_501c3,
        "grant": grant_kw,
        "funding": funding_kw,
        "relevence_label": "",
        "grant_yesno": "grant",
        "label_note": "",
        "keyword_score": k_score,
        "cosine_similarity": round(float(c_score or 0.0), 6),
        "granting_organization": "",
        "date_of_loi_due": "",
        "date_of_submission": "",
        "doc_id": did,
        "source_domain": urlparse(url).netloc,
        "html_path": "",      # not saving html in demo mode
        "content_sha1": sha1,
        "is_detail_page": 1,
        "text_path": "",      # not saving text in demo mode
        "date_seen_utc": date_seen,
        "svm_mission_score": float(svm_score),
    }
    return row

# ============================================================
# crawl a single site (BFS) with per-site caps + global cap
# ============================================================
def crawl_site(seed_url, svm_vec, svm_clf, rows, visited_global,
               per_site_target=PER_SITE_TARGET,
               max_depth=MAX_DEPTH,
               max_site_visits=MAX_SITE_VISITS):
    """
    Crawl one domain starting from seed_url and append rows in-place.
    Stops after:
      - collecting per_site_target SVM-passing rows for this site, OR
      - visiting max_site_visits pages for this site, OR
      - reaching global MAX_DEMO_ROWS, OR
      - running out of links.
    """
    queue = [(seed_url, 0)]
    base_url = seed_url
    site_rows = 0
    site_visits = 0
    site_host = urlparse(seed_url).netloc.lower()

    while queue and site_rows < per_site_target and site_visits < max_site_visits:
        if len(rows) >= MAX_DEMO_ROWS:
            print(f"\n******* GLOBAL STOP: collected {len(rows)} demo rows *******\n")
            break

        url, depth = queue.pop(0)

        if url in visited_global:
            continue
        visited_global.add(url)

        if is_denied_url(url):
            print(f"SKIP denied: {url}")
            continue

        if not same_domain(url, base_url):
            print(f"SKIP other domain: {url}")
            continue

        site_visits += 1
        print(f"\nVISIT (depth={depth}, visits={site_visits}, "
              f"site_rows={site_rows}, total_rows={len(rows)}): {url}")

        html = fetch_url(url)
        time.sleep(random.uniform(0.8, 1.5))

        if html is None:
            continue

        clean_text, soup = html_to_clean_text(html)
        if not clean_text:
            continue

        # 1) LR grant/not-grant
        is_gr, lr_prob = is_grant_by_lr(clean_text)
        print(f"  LR prob(grant)={round(lr_prob,3)} -> is_grant={is_gr}")

        mission_ok = False
        k_score = 0
        c_score = 0.0
        matched = []

        if is_gr:
            # 2) keyword + cosine mission filter
            try:
                k_score, matched = keyword_score(clean_text, keyword_weights)
                c_score = cosine_score(clean_text)
            except Exception as e:
                print(f"  error computing keyword/cosine for {url}: {e}")
                k_score, matched, c_score = 0, [], 0.0

            mission_ok = (k_score >= 7) or (k_score > 0 and c_score >= 0.07)

            print(
                f"  keyword_score={k_score}, matched={matched}, "
                f"cosine={round(c_score,3)}, mission_ok={mission_ok}"
            )

        if is_gr and mission_ok:
            # 3) final SVM mission score
            svm_score = compute_svm_score(clean_text, svm_vec, svm_clf)
            print(f"  SVM mission score={round(svm_score,3)}")

            if svm_score < SVM_THRESHOLD:
                print(f"  SVM filter: score {round(svm_score,3)} < {SVM_THRESHOLD} -> DROP")
            else:
                scrape_num = len(rows) + 1
                row = build_demo_row(
                    url=url,
                    clean_text=clean_text,
                    k_score=k_score,
                    matched_list=matched,
                    c_score=c_score,
                    svm_score=svm_score,
                    scrape_num=scrape_num,
                )
                rows.append(row)
                site_rows += 1
                print(f"  --> SAVED ROW #{site_rows} for {site_host} "
                      f"(total_rows={len(rows)})")

                if len(rows) >= MAX_DEMO_ROWS:
                    print(f"\n******* GLOBAL STOP: collected {len(rows)} demo rows *******\n")
                    break

        # expand links if we still have depth and visit budget
        if depth < max_depth and site_visits < max_site_visits:
            child_links = extract_links(url, soup, base_url)
            print(f"  found {len(child_links)} child links")
            for nxt in child_links:
                if nxt not in visited_global and nxt not in [u for (u, _) in queue]:
                    queue.append((nxt, depth + 1))

    print(f"Finished site {site_host}: "
          f"collected {site_rows} rows, visited {site_visits} pages")

# ============================================================
# run demo across all seeds (static + search)
# ============================================================
def run_demo():
    print("\nLoading SVM + LR models...")
    svm_vec, svm_clf = load_svm_model()

    rows = []
    visited_global = set()

    # 1) crawl static seeds
    for seed in static_urls:
        if len(rows) >= MAX_DEMO_ROWS:
            break
        print(f"\n========== CRAWLING SEED (STATIC): {seed} ==========")
        crawl_site(
            seed_url=seed,
            svm_vec=svm_vec,
            svm_clf=svm_clf,
            rows=rows,
            visited_global=visited_global,
            per_site_target=PER_SITE_TARGET,
            max_depth=MAX_DEPTH,
            max_site_visits=MAX_SITE_VISITS,
        )

    # 2) discovery via web search
    for query in search_keywords:
        if len(rows) >= MAX_DEMO_ROWS:
            break

        print(f"\nSEARCHING WEB FOR: '{query}'")
        try:
            found_urls = search_web(query, max_results=30)
        except Exception as e:
            print(f"search error for '{query}': {e}")
            found_urls = []

        for url in found_urls:
            if len(rows) >= MAX_DEMO_ROWS:
                break
            print(f"\n========== CRAWLING SEED (SEARCH): {url} ==========")
            crawl_site(
                seed_url=url,
                svm_vec=svm_vec,
                svm_clf=svm_clf,
                rows=rows,
                visited_global=visited_global,
                per_site_target=PER_SITE_TARGET,
                max_depth=MAX_DEPTH,
                max_site_visits=MAX_SITE_VISITS,
            )
            time.sleep(0.5)

    if not rows:
        print("No rows collected; nothing to write.")
        return

    df = pd.DataFrame(rows)
    df.to_csv(DEMO_CSV_PATH, index=False, encoding="utf-8")
    print(f"\nWrote demo CSV with {len(rows)} rows to: {DEMO_CSV_PATH}")


if __name__ == "__main__":
    run_demo()


Loaded grant LR model bundle from C:\Users\miked\Desktop2\IConnectFoundation\grantMinded\grant_logreg_bundle.pkl

Loading SVM + LR models...
Loaded relevance_vectorizer + relevance_svm_best from C:\Users\miked\Desktop2\IConnectFoundation\grantMinded

========== CRAWLING SEED (STATIC): https://www.rwjf.org/en/grants/active-funding-opportunities.html ==========

VISIT (depth=0, visits=1, site_rows=0, total_rows=0): https://www.rwjf.org/en/grants/active-funding-opportunities.html
  LR prob(grant)=0.879 -> is_grant=True
  keyword_score=6, matched=['grant', 'funding'], cosine=0.097, mission_ok=True
  SVM mission score=-1.088
  SVM filter: score -1.088 < 0.0 -> DROP
  found 15 child links

VISIT (depth=1, visits=2, site_rows=0, total_rows=0): https://www.rwjf.org/en/grants.html
  LR prob(grant)=0.71 -> is_grant=True
  keyword_score=6, matched=['grant', 'funding'], cosine=0.13, mission_ok=True
  SVM mission score=-0.866
  SVM filter: score -0.866 < 0.0 -> DROP
  found 17 child links

VISIT (d